# Facial Recognition System

## 1. Environment and Dependencies
This notebook requires Python 3.7 and the dependencies listed in the provided `environment.yml` file.

The recommended installation method is:

```bash
conda env create -f environment.yml
conda activate facial-recognition
```
Alternatively, the required packages can be installed manually using the instructions in the README.md.

----------------------------------------------------------------------------------------------------------

<div class="alert alert-block alert-info">
<b>IMPORTANT:</b> Before running this notebook, make sure the environment is configured and follow the data download and setup instructions provided in the `README.md`.
</div>

### 1.1 Import Dependencies

In [1]:
# Import standard dependencies
import cv2
import os
import random
import numpy as np
from matplotlib import pyplot as plt

In [2]:
# Set to 0 to show all TensorFlow logs while debugging.
# Keep at 3 to suppress INFO, WARNING, and ERROR startup messages.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [3]:
# Import tensorflow dependencies - Functional API
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
import tensorflow as tf

### 1.2 Verify the Environment

The following cell verifies that the main dependencies are installed correctly and checks whether TensorFlow can detect and use the GPU.
This verification step is optional and is included only to help diagnose installation, dependency, or GPU configuration issues. It is not required for the main facial recognition workflow.

In [4]:
# import sys

# print("=== Environment Check ===")
# print("Python:", sys.version)
# print("Executable:", sys.executable)

# print("\n=== NumPy ===")
# try:
#     import numpy as np

#     print("NumPy version:", np.__version__)
#     print("NumPy test:", np.array([1, 2, 3]) * 2)
# except Exception as error:
#     print("NumPy FAILED:", error)

# print("\n=== TensorFlow ===")
# try:
#     import tensorflow as tf

#     print("TensorFlow version:", tf.__version__)
#     print("Built with CUDA:", tf.test.is_built_with_cuda())

#     gpus = tf.config.list_physical_devices("GPU")
#     print("Detected GPUs:", gpus)

#     a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
#     b = tf.constant([[5.0, 6.0], [7.0, 8.0]])

#     device = "/GPU:0" if gpus else "/CPU:0"

#     with tf.device(device):
#         result = tf.matmul(a, b)

#     print("TensorFlow result:")
#     print(result.numpy())
#     print("Device used:", result.device)

#     if gpus and "GPU" in result.device.upper():
#         print("GPU test: WORKING")
#     elif gpus:
#         print("GPU detected, but this operation did not run on it.")
#     else:
#         print("GPU test: NOT DETECTED")

# except Exception as error:
#     print("TensorFlow FAILED:", error)

# print("\n=== OpenCV ===")
# try:
#     import cv2

#     print("OpenCV version:", cv2.__version__)
#     print("OpenCV test: WORKING")
# except Exception as error:
#     print("OpenCV FAILED:", error)

# print("\n=== Matplotlib ===")
# try:
#     import matplotlib

#     print("Matplotlib version:", matplotlib.__version__)
#     print("Matplotlib test: WORKING")
# except Exception as error:
#     print("Matplotlib FAILED:", error)

# print("\n=== Jupyter ===")
# try:
#     import notebook
#     import ipykernel

#     print("Notebook version:", notebook.__version__)
#     print("ipykernel version:", ipykernel.__version__)
#     print("Jupyter test: WORKING")
# except Exception as error:
#     print("Jupyter FAILED:", error)

# print("\n=== Check Complete ===")

### 1.3 Set GPU Growth

In [5]:
# Avoid OOM (out of memory errors) errors by setting GPU Memory Consumption Growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus: 
    tf.config.experimental.set_memory_growth(gpu, True)

In [6]:
# Uncomment the lines below to verify that TensorFlow detects the GPU
# print(gpus)
# print(tf.config.list_logical_devices('GPU'))

### 1.4 Create Folder Structures

In [7]:
# Setup paths
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [8]:
# Make the directories
os.makedirs(POS_PATH)
os.makedirs(NEG_PATH)
os.makedirs(ANC_PATH)

## 2. Collect Positives and Anchors

### 2.1 Untar Labelled Faces in the Wild Dataset

In [ ]:
# https://www.kaggle.com/datasets/atulanandjha/lfwpeople
# Download the LFW (Labeled Faces in the Wild) dataset archive.
# Save the downloaded folder in the same folder as the jupyter notebook

In [10]:
# Extract the downloaded LFW dataset archive into the current project folder.
import zipfile

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall(".")

print("Dataset extracted successfully.")

Dataset extracted successfully.


In [11]:
# Check the extracted files and folders: we expect to see: lfw-funneled.tgz
print(os.listdir("."))

['archive.zip:Zone.Identifier', 'Facial Verification with a Siamese Network.ipynb', 'archive.zip', 'pairs.txt', '.ipynb_checkpoints', 'pairsDevTest.txt', 'pairsDevTrain.txt', 'data', 'lfw-funneled.tgz']


In [12]:
# Extract the LFW .tgz archive.
import tarfile

with tarfile.open("lfw-funneled.tgz", "r:gz") as tar:
    tar.extractall(".")

print("LFW dataset extracted successfully.")

LFW dataset extracted successfully.


In [13]:
# Check the extracted dataset folder: we expect to see lfw_funneled added
print(os.listdir("."))

['archive.zip:Zone.Identifier', 'lfw_funneled', 'Facial Verification with a Siamese Network.ipynb', 'archive.zip', 'pairs.txt', '.ipynb_checkpoints', 'pairsDevTest.txt', 'pairsDevTrain.txt', 'data', 'lfw-funneled.tgz']


In [15]:
# set path
LFW_PATH = "lfw_funneled"

In [16]:
# Move all LFW images into data/negative.
for directory in os.listdir(LFW_PATH):
    directory_path = os.path.join(LFW_PATH, directory)

    if os.path.isdir(directory_path):
        for file in os.listdir(directory_path):
            old_path = os.path.join(directory_path, file)
            new_path = os.path.join(NEG_PATH, file)
            os.replace(old_path, new_path)

### 2.2 Collect Positive and Anchor Classes

In [1]:
import os

print(os.listdir("/dev") if os.path.exists("/dev") else "No /dev directory")
print("Camera devices:", [
    device for device in os.listdir("/dev")
    if device.startswith("video")
])

['media0', 'video1', 'video0', 'bus', 'shm', 'pts', 'stderr', 'stdout', 'stdin', 'fd', 'block', 'sdd', 'sg3', 'net', 'sdc', 'sg2', 'cpu_dma_latency', 'sdb', 'mcelog', 'sda', 'sg1', 'bsg', 'vsock', 'sg0', 'dxg', 'ptp0', 'mapper', 'rtc0', 'mptctl', 'ppp', 'loop7', 'loop6', 'loop5', 'loop4', 'loop3', 'loop2', 'loop1', 'loop0', 'loop-control', 'ram15', 'ram14', 'ram13', 'ram12', 'ram11', 'ram10', 'ram9', 'ram8', 'ram7', 'ram6', 'ram5', 'ram4', 'ram3', 'ram2', 'ram1', 'ram0', 'vport0p2', 'vport0p1', 'hvc7', 'hvc6', 'hvc5', 'hvc4', 'hvc3', 'hvc2', 'hvc1', 'hvc0', 'vport0p0', 'hpet', 'ttyS7', 'ttyS6', 'ttyS5', 'ttyS4', 'ttyS3', 'ttyS2', 'ttyS1', 'ttyS0', 'ptmx', 'fuse', 'userfaultfd', 'snapshot', 'hwrng', 'tty63', 'tty62', 'tty61', 'tty60', 'tty59', 'tty58', 'tty57', 'tty56', 'tty55', 'tty54', 'tty53', 'tty52', 'tty51', 'tty50', 'tty49', 'tty48', 'tty47', 'tty46', 'tty45', 'tty44', 'tty43', 'tty42', 'tty41', 'tty40', 'tty39', 'tty38', 'tty37', 'tty36', 'tty35', 'tty34', 'tty33', 'tty32', 'tty

In [2]:
import cv2

cap = cv2.VideoCapture("/dev/video0", cv2.CAP_V4L2)

cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 10)

if not cap.isOpened():
    raise RuntimeError("Could not open /dev/video0.")

ret, frame = cap.read()

if not ret:
    raise RuntimeError("Camera opened, but the frame was corrupted or unavailable.")

cv2.imwrite("webcam-test.jpg", frame)
cap.release()

print("Captured webcam-test.jpg")

Captured webcam-test.jpg


## 3. Load and Preprocess Images

### 3.1 Get Image Directories

### 3.2 Preprocessing - Scale and Resize

### 3.3 Create Labelled Dataset

### 3.4 Build Train and Test Partition

## 4. Model Engineering

### 4.1 Build Embedding Layer

### 4.2 Build Distance Layer

### 4.3 Make Siamese Model

## 5. Training

### 5.1 Setup Loss and Optimizer

### 5.2 Establish Checkpoints

### 5.3 Build Train Step Function

### 5.4 Build Training Loop

### 5.5 Train the Model

## 6. Evaluate Model

### 6.1 Import Metrics

### 6.2 Make Predictions

### 6.3 Calculate Metrics

### 6.4 Visualize Results

## 7. Save Model

## 8. Real-Time Test

### 8.1 Verification Function

### 8.2 OpenCV Real-Time Verification